# RPG Eval-Seed Result Tables

This lightweight notebook loads `summary.json` files from `artifacts/rpg/eval_seeds` and builds reporting tables for all available datasets and config tracks.

Expected result layout:

```text
artifacts/rpg/eval_seeds/<track>/<dataset>/<session>/summary.json
```

where `<track>` is usually `released_readme` or `paper_appendix`.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

ROOT = Path.cwd()
ARTIFACT_ROOT = ROOT / "artifacts" / "rpg" / "eval_seeds"

# If you run this notebook from another working directory, uncomment and set manually.
# ARTIFACT_ROOT = Path("/gpfs/home6/$USER/RPG/artifacts/rpg/eval_seeds").expanduser()

ARTIFACT_ROOT

In [ ]:
TRACK_LABELS = {
    "released_readme": "released README",
    "paper_appendix": "paper appendix",
}

DATASET_LABELS = {
    "Sports_and_Outdoors": "Sports",
    "sports_and_outdoors": "Sports",
    "Beauty": "Beauty",
    "beauty": "Beauty",
    "Toys_and_Games": "Toys",
    "toys_and_games": "Toys",
    "CDs_and_Vinyl": "CDs",
    "cds_and_vinyl": "CDs",
}


def _metric_summary(payload: dict, metric: str = "ndcg@10") -> dict:
    for row in payload.get("metric_summary", []):
        if row.get("metric") == metric:
            return row
    raise KeyError(f"Metric {metric!r} not found in metric_summary")


def _mean_visited_items(payload: dict) -> float:
    values = [row.get("n_visited_items") for row in payload.get("per_seed_summary", [])]
    values = [float(value) for value in values if value is not None]
    return float(np.mean(values)) if values else math.nan


def _track_and_dataset(summary_path: Path) -> tuple[str, str]:
    rel_parts = summary_path.relative_to(ARTIFACT_ROOT).parts
    if len(rel_parts) >= 4 and rel_parts[0] in TRACK_LABELS:
        return rel_parts[0], rel_parts[1]
    if len(rel_parts) >= 3:
        return "unknown", rel_parts[0]
    return "unknown", "unknown"


def load_eval_seed_runs(metric: str = "ndcg@10") -> pd.DataFrame:
    if not ARTIFACT_ROOT.exists():
        return pd.DataFrame()

    rows = []
    for summary_path in sorted(ARTIFACT_ROOT.rglob("summary.json")):
        payload = json.loads(summary_path.read_text())
        track, dataset_slug = _track_and_dataset(summary_path)
        metric_row = _metric_summary(payload, metric=metric)
        paper_value = metric_row.get("paper_value")
        ours = float(metric_row["final_user_avg"])
        diff = None if paper_value is None else ours - float(paper_value)
        eval_seed_std = float(metric_row.get("eval_seed_std", math.nan))
        diff_over_seed_std = math.nan
        if diff is not None and eval_seed_std > 0:
            diff_over_seed_std = abs(diff) / eval_seed_std

        ci_low = float(metric_row.get("user_bootstrap_ci_low", math.nan))
        ci_high = float(metric_row.get("user_bootstrap_ci_high", math.nan))
        paper_inside_ci = None
        if paper_value is not None and not math.isnan(ci_low) and not math.isnan(ci_high):
            paper_inside_ci = ci_low <= float(paper_value) <= ci_high

        rows.append(
            {
                "track": track,
                "config_source": TRACK_LABELS.get(track, track),
                "dataset_slug": dataset_slug,
                "dataset": DATASET_LABELS.get(payload.get("category"), DATASET_LABELS.get(dataset_slug, dataset_slug)),
                "session": summary_path.parent.name,
                "summary_path": str(summary_path),
                "checkpoint_path": payload.get("checkpoint_path"),
                "n_eval_seeds": len(payload.get("eval_seeds", [])),
                "n_users": int(metric_row.get("n_users", 0)),
                "metric": metric,
                "paper_ndcg10": None if paper_value is None else float(paper_value),
                "ours_ndcg10": ours,
                "diff": diff,
                "eval_seed_std": eval_seed_std,
                "diff_over_seed_std": diff_over_seed_std,
                "equivalent_pm001": metric_row.get("equivalent_to_paper"),
                "user_ci_level": float(metric_row.get("user_bootstrap_ci_level", math.nan)),
                "user_ci_low": ci_low,
                "user_ci_high": ci_high,
                "paper_inside_user_ci": paper_inside_ci,
                "mean_visited_items": _mean_visited_items(payload),
                "bootstrap_samples": int(payload.get("bootstrap_samples", 0)),
                "bootstrap_seed": int(payload.get("bootstrap_seed", 0)),
            }
        )

    return pd.DataFrame(rows)


all_runs = load_eval_seed_runs("ndcg@10")
all_runs

In [ ]:
if all_runs.empty:
    print(f"No summary.json files found under {ARTIFACT_ROOT}")
else:
    # Keep the newest-looking session for each dataset/config source.
    latest = (
        all_runs.sort_values(["dataset", "track", "session"])
        .groupby(["dataset", "track"], as_index=False)
        .tail(1)
        .sort_values(["dataset", "track"])
        .reset_index(drop=True)
    )
    display(latest[["dataset", "config_source", "session", "summary_path"]])

## Table 1: Reproduction Accuracy

This table focuses on fixed-split reproduction: how far our `NDCG@10` is from the paper and whether graph-decoding eval-seed variance plausibly explains the gap.

In [ ]:
if not all_runs.empty:
    table_reproduction = latest[
        [
            "dataset",
            "config_source",
            "paper_ndcg10",
            "ours_ndcg10",
            "diff",
            "eval_seed_std",
            "diff_over_seed_std",
            "equivalent_pm001",
        ]
    ].copy()
    table_reproduction = table_reproduction.rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "paper_ndcg10": "Paper NDCG@10",
            "ours_ndcg10": "Ours NDCG@10",
            "diff": "Diff",
            "eval_seed_std": "Eval seed std",
            "diff_over_seed_std": "|Diff| / seed std",
            "equivalent_pm001": "Equivalent ±0.001?",
        }
    )
    display(table_reproduction.style.format({
        "Paper NDCG@10": "{:.4f}",
        "Ours NDCG@10": "{:.5f}",
        "Diff": "{:+.5f}",
        "Eval seed std": "{:.6f}",
        "|Diff| / seed std": "{:.1f}",
    }))

## Table 2: User-Level Uncertainty

This table reports the classical IR-style uncertainty over users. It should be interpreted separately from eval-seed variance.

In [ ]:
if not all_runs.empty:
    table_user_ci = latest[
        [
            "dataset",
            "config_source",
            "ours_ndcg10",
            "user_ci_low",
            "user_ci_high",
            "paper_inside_user_ci",
            "bootstrap_samples",
            "bootstrap_seed",
        ]
    ].copy()
    table_user_ci["95% user CI"] = table_user_ci.apply(
        lambda row: f"[{row['user_ci_low']:.5f}, {row['user_ci_high']:.5f}]",
        axis=1,
    )
    table_user_ci = table_user_ci[
        ["dataset", "config_source", "ours_ndcg10", "95% user CI", "paper_inside_user_ci", "bootstrap_samples", "bootstrap_seed"]
    ].rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "ours_ndcg10": "Ours NDCG@10",
            "paper_inside_user_ci": "Paper inside CI?",
            "bootstrap_samples": "Bootstrap samples",
            "bootstrap_seed": "Bootstrap seed",
        }
    )
    display(table_user_ci.style.format({"Ours NDCG@10": "{:.5f}"}))

## Appendix Table: Full Run Metadata

Useful for checking which checkpoint/session produced each row.

In [ ]:
if not all_runs.empty:
    metadata_cols = [
        "dataset",
        "config_source",
        "session",
        "n_eval_seeds",
        "n_users",
        "mean_visited_items",
        "checkpoint_path",
        "summary_path",
    ]
    display(latest[metadata_cols].style.format({"mean_visited_items": "{:.1f}"}))